In [1]:
# Parameters
BATCH_MODE = "true"


# 5. RAG Modern - Retrieval Augmented Generation

**Navigation** : [Index](README.md) | [<< Précédent](04_Function_Calling.ipynb) | [Suivant >>](06_PDF_Web_Search.ipynb)


**Durée estimée** : 65 minutes

**Prérequis** : Notebooks 1 (OpenAI Intro), 4 (Function Calling)

***

## Objectifs

La **RAG** (Retrieval Augmented Generation) permet d'enrichir les réponses d'un LLM avec des données externes (documents, bases de connaissances). Ce notebook couvre :

1. **Fondamentaux RAG** : Embeddings, chunking, recherche vectorielle
2. **Stratégies de chunking** : Fixe, sémantique, récursif
3. **Responses API** : Multi-turn RAG avec `previous_response_id`
4. **Mesure d'usage** : Comparaison des champs `usage` renvoyés à chaque tour
5. **Citations et sources** : Traçabilité des réponses

***

## Pourquoi la RAG ?

| Problème LLM seul | Solution RAG |
|-------------------|-------------|
| Connaissances figées (cutoff date) | Données actualisées en temps réel |
| Hallucinations fréquentes | Réponses basées sur sources vérifiables |
| Pas de données privées | Accès à vos documents internes |
| Contexte limité | Fenêtre étendue via retrieval |

## Configuration

In [2]:
# Dépendances pré-provisionnées (openai, tiktoken, python-dotenv, scikit-learn, numpy, pandas, requests, beautifulsoup4, lxml) : voir GenAI/requirements.txt
import openai, tiktoken, dotenv, sklearn, numpy, pandas, requests, bs4, lxml
print("Dépendances RAG pré-chargées avec succès.")

Dépendances RAG pré-chargées avec succès.


Les dépendances étant installées, le client OpenAI et les variables de configuration sont initialisés. La détection du fichier `.env` garantit que la clé API est chargée quel que soit le répertoire de travail courant.

In [3]:
# Import guards - verification des dependances
try:
    import openai
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False

try:
    import anthropic
    ANTHROPIC_AVAILABLE = True
except ImportError:
    ANTHROPIC_AVAILABLE = False

try:
    import requests
    REQUESTS_AVAILABLE = True
except ImportError:
    REQUESTS_AVAILABLE = False

try:
    import matplotlib
    MATPLOTLIB_AVAILABLE = True
except ImportError:
    MATPLOTLIB_AVAILABLE = False

try:
    import numpy
    NUMPY_AVAILABLE = True
except ImportError:
    NUMPY_AVAILABLE = False

try:
    import pandas
    PANDAS_AVAILABLE = True
except ImportError:
    PANDAS_AVAILABLE = False

try:
    from PIL import Image
    PIL_AVAILABLE = True
except ImportError:
    PIL_AVAILABLE = False

try:
    import IPython
    IPYTHON_AVAILABLE = True
except ImportError:
    IPYTHON_AVAILABLE = False

try:
    from dotenv import load_dotenv
    DOTENV_AVAILABLE = True
except ImportError:
    DOTENV_AVAILABLE = False

try:
    import torch
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False

try:
    import transformers
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    TRANSFORMERS_AVAILABLE = False

try:
    import cv2
    CV2_AVAILABLE = True
except ImportError:
    CV2_AVAILABLE = False

try:
    import pydantic
    PYDANTIC_AVAILABLE = True
except ImportError:
    PYDANTIC_AVAILABLE = False

# Resume des dependances disponibles
_DEPS = {
    'openai': OPENAI_AVAILABLE,
    'anthropic': ANTHROPIC_AVAILABLE,
    'requests': REQUESTS_AVAILABLE,
    'matplotlib': MATPLOTLIB_AVAILABLE,
    'numpy': NUMPY_AVAILABLE,
    'pandas': PANDAS_AVAILABLE,
    'PIL': PIL_AVAILABLE,
    'IPython': IPYTHON_AVAILABLE,
    'dotenv': DOTENV_AVAILABLE,
    'torch': TORCH_AVAILABLE,
    'transformers': TRANSFORMERS_AVAILABLE,
    'cv2': CV2_AVAILABLE,
    'pydantic': PYDANTIC_AVAILABLE,
}
available = [k for k, v in _DEPS.items() if v]
missing = [k for k, v in _DEPS.items() if not v]
print(f"Dependances: {len(available)}/{len(_DEPS)} disponibles"
      + (f" | Manquantes: {missing}" if missing else ""))

from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
from typing import List, Dict, Any
from dotenv import load_dotenv
from openai import OpenAI

# Charger les variables d'environnement
# Chargement robuste de la configuration .env
from dotenv import load_dotenv
import os
# Recherche du .env dans tous les parents (pour Papermill qui change le cwd)
current_path = Path.cwd()
env_loaded = False
for _ in range(10):
    env_path = current_path / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env charge depuis: {env_path.name}")
        env_loaded = True
        break
    if current_path.name == "GenAI" or len(current_path.parts) <= 1:
        break
    current_path = current_path.parent
if not env_loaded:
    print("WARNING: .env non trouve, utilisation variables environnement")

# Mode batch pour tests automatisés
BATCH_MODE = os.getenv("BATCH_MODE", "false").lower() == "true"

# Client OpenAI
client = OpenAI()

# Modèle par défaut depuis .env
DEFAULT_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-mini")

print(f"Configuration chargée - Mode batch: {BATCH_MODE}")
print(f"Modèle embeddings: text-embedding-3-large")
print(f"Modèle génération: {DEFAULT_MODEL}")

Dependances: 10/13 disponibles | Manquantes: ['torch', 'transformers', 'cv2']
.env charge depuis: .env
Configuration chargée - Mode batch: False
Modèle embeddings: text-embedding-3-large
Modèle génération: gpt-5-mini


### Interprétation de la configuration

La cellule précédente initialise l'environnement RAG avec plusieurs composants clés.

**Configuration validée** :

| Composant | Statut | Valeur |
|-----------|--------|--------|
| **Client OpenAI** | ✓ Initialisé | Authentification via `OPENAI_API_KEY` dans `.env` |
| **Mode batch** | Variable | `BATCH_MODE` depuis `.env` (default: `false`) |
| **Modèle génération** | Configurable | `OPENAI_MODEL` depuis `.env` (default: `gpt-5-mini`) |
| **Modèle embeddings** | Fixe | `text-embedding-3-large` (hardcodé pour précision) |

**Choix techniques justifiés** :

1. **`text-embedding-3-large`** : Modèle d'embedding le plus performant d'OpenAI
   - 3072 dimensions vs 1536 pour `-small`
   - Meilleure capture des nuances sémantiques
   - Coût : ~0.13$ pour 1M tokens (acceptable pour RAG)

2. **Mode batch** : Permet l'exécution automatisée (Papermill, CI/CD)
   - `BATCH_MODE=true` : Skip les widgets interactifs
   - Utile pour tests et validation

3. **Modèle par défaut** : `gpt-5-mini`
   - Rapide et économique pour la génération
   - Peut être remplacé par `gpt-5` pour plus de précision si besoin

**Variables d'environnement requises** (`.env`) :
```bash
OPENAI_API_KEY=sk-...           # Obligatoire
OPENAI_MODEL=gpt-5-mini         # Optionnel (default)
BATCH_MODE=false                # Optionnel (default)
```

**Point important** : L'initialisation du `client` OpenAI échouera si `OPENAI_API_KEY` n'est pas définie. Assurez-vous d'avoir copié `.env.example` vers `.env` avec vos vraies clés.

### Bibliothèques utilisées

Ce notebook nécessite plusieurs dépendances clés :

- **openai** : API OpenAI pour embeddings et génération
- **tiktoken** : Tokenizer officiel pour compter les tokens
- **scikit-learn** : k-Nearest Neighbors pour la recherche vectorielle
- **numpy/pandas** : Manipulation de données et vecteurs
- **requests/beautifulsoup4** : Scraping web pour récupérer le document source
- **python-dotenv** : Gestion sécurisée des clés API

**Note** : En production, remplacer scikit-learn par une vraie base vectorielle (Pinecone, Qdrant, Weaviate).

***

## Partie 1 : Fondamentaux RAG

### 1.1 Architecture RAG

> **Ancrage scholarly.** Le terme **RAG** (*Retrieval-Augmented Generation*) a été
> introduit par **Lewis et al. (2020)** -- *Retrieval-Augmented Generation for
> Knowledge-Intensive NLP Tasks* (arXiv:2005.11401) -- qui proposent d'apprendre un
> retriever et un générateur de façon jointe pour réduire les hallucinations sur les
> tâches connaissance-intensives. Le notebook ci-dessous en implémente une variante
> moderne (embeddings pré-entraînés + LLM gelé).

```
┌─────────────┐     ┌──────────────┐     ┌─────────────┐
│  Documents  │ ──► │   Chunking   │ ──► │  Embeddings │
└─────────────┘     └──────────────┘     └──────┬──────┘
                                                │
                                                ▼
┌─────────────┐     ┌──────────────┐     ┌─────────────┐
│   Réponse   │ ◄── │     LLM      │ ◄── │  Retrieval  │
└─────────────┘     └──────────────┘     └─────────────┘
                           ▲
                           │
                    ┌──────┴──────┐
                    │   Question  │
                    └─────────────┘
```

**Étapes** :
1. **Indexation** : Documents → Chunks → Embeddings → Base vectorielle
2. **Retrieval** : Question → Embedding → k-NN → Chunks pertinents
3. **Generation** : Question + Chunks → LLM → Réponse augmentée

Le diagramme ci-dessous rend ce pipeline RAG sous forme de graphe : la phase
d'**indexation** (Documents -> Retrieval) et la phase de **requête/génération**
(Question + Retrieval -> LLM -> Reponse).

```mermaid
flowchart LR
    D["Documents"] --> CH["Chunking"] --> EMB["Embeddings"] --> R["Retrieval"]
    Q["Question"] --> LLM["LLM"]
    R --> LLM
    LLM --> REP(["Reponse"])
    classDef idx fill:#cfe2ff,stroke:#084298,color:#052c65
    classDef gen fill:#fff3cd,stroke:#b8860b,color:#5c4400
    classDef out fill:#d1e7dd,stroke:#0f5132,color:#0a3622
    class D,CH,EMB,R idx
    class Q,LLM gen
    class REP out
```

> **Lecture.** En amont (hors ligne), les **documents** sont decoupes en *chunks*,
> encodes en **embeddings** et indexes pour le **retrieval**. A la requête, la
> **question** declenche une recherche des passages pertinents, et le **LLM** synthetise
> la **reponse** en s'appuyant sur ces passages -- c'est l'apport du RAG : ancrer la
> génération dans des sources externes plutôt que dans la seule memoire parametrique.


## 1.2 Préparation des données

Pour cet exemple, nous utilisons le débat Lincoln-Douglas (1858), un document historique public.

### Pourquoi le débat Lincoln-Douglas ?

Ce document historique est un **excellent cas d'usage RAG** :

1. **Domaine public** : Accessible librement, pas de problèmes de copyright
2. **Taille idéale** : ~98k caractères, assez long pour nécessiter du chunking
3. **Structure narrative** : Discours structurés avec arguments clairs
4. **Questions naturelles** : "Quelle était la position de Lincoln ?" → Réponses vérifiables

**Fallback pour les tests** : Le code inclut un texte de secours si le scraping échoue (mode batch, timeout réseau, etc.). Cela garantit que le notebook peut s'exécuter même sans connexion.

**Point technique** : Nous récupérons uniquement le premier débat (Ottawa, 21 août 1858). La série complète comprend 7 débats.

In [4]:
import requests
from bs4 import BeautifulSoup

def fetch_debate_text() -> str:
    """Récupère le texte du premier débat Lincoln-Douglas."""
    url = "https://home.nps.gov/liho/learn/historyculture/debate1.htm"
    
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, "html.parser")
        main_div = soup.select_one("div.ColumnMain")
        
        if main_div:
            return main_div.get_text(separator="\n", strip=True)
        else:
            raise ValueError("Conteneur principal non trouvé")
            
    except Exception as e:
        print(f"Erreur lors du scraping: {e}")
        # Texte de fallback pour le mode batch
        return """Lincoln-Douglas Debate - Ottawa, Illinois, August 21, 1858
        
Abraham Lincoln argued that slavery was morally wrong and should not be extended 
into new territories. He believed in the principle that "all men are created equal" 
as stated in the Declaration of Independence.

Lincoln stated: "I have no purpose to introduce political and social equality 
between the white and black races. There is a physical difference between the two, 
which, in my judgment, will probably forever forbid their living together upon the 
footing of perfect equality."

However, Lincoln maintained that Black Americans had the right to earn their own 
bread and improve their condition. He opposed the expansion of slavery while 
acknowledging the constitutional protections it had in existing states.

Douglas advocated for popular sovereignty, allowing each territory to decide 
the slavery question for itself. He accused Lincoln of wanting to make all 
states free states, which Lincoln denied.

The debate centered on the interpretation of the Dred Scott decision and 
whether Congress could prohibit slavery in the territories."""

# Récupération du texte
debate_text = fetch_debate_text()
print(f"Texte récupéré: {len(debate_text)} caractères")
print(f"Aperçu: {debate_text[:500]}...")

Texte récupéré: 98529 caractères
Aperçu: First Debate: Ottawa, Illinois
August 21, 1858
It was dry and dusty, between 10,000 and 12,000 people were in attendance when the debate began at 2:00 p.m. There were no seats or bleachers.
Douglas charged Lincoln with trying to “abolitionize” the Whig and Democratic Parties. He also charged Lincoln had been present when a very radical “abolitionist” type platform had been written by the Republican Party in 1854. Douglas accused Lincoln of taking the side of the common enemy in the Mexican War. ...


### Analyse du document récupéré

Le scraping a réussi et notre base de données source est prête pour le chunking.

**Caractéristiques du document** :

| Aspect | Valeur | Interprétation |
|--------|--------|----------------|
| **Taille** | 98,529 caractères | Document substantiel, nécessite chunking |
| **Source** | NPS.gov (National Park Service) | Source fiable, domaine public |
| **Date** | 21 août 1858 (Ottawa, Illinois) | Premier des 7 débats Lincoln-Douglas |
| **Audience estimée** | 10,000-12,000 personnes | Événement historique majeur |

**Aperçu du contenu** :

Le texte commence directement avec les accusations de Douglas contre Lincoln :
- "Abolitionize" les partis politiques
- Plateforme "abolitionniste radicale" de 1854
- Position controversée sur la guerre du Mexique

**Stratégie de chunking recommandée** :

Pour un document narratif de cette taille (~98k caractères), nous utiliserons :
- **Chunking fixe** : 400 mots avec overlap de 50
- **Raison** : Le débat suit une structure de discours/réponse linéaire
- **Résultat attendu** : ~50 chunks de taille équilibrée

**Alternative avec fallback** : Le code inclut un texte de secours (mode batch/timeout réseau) qui garantit l'exécution même sans connexion. C'est une **bonne pratique** pour les notebooks pédagogiques.

**Prochaine étape** : Découper ce texte en chunks optimisés pour la recherche sémantique.

### Implémentation des stratégies de chunking

Les trois fonctions suivantes implémentent les stratégies présentées ci-dessus. Chacune a ses avantages selon le type de document.

***

## Partie 2 : Stratégies de Chunking

Le chunking est **critique** pour la qualité de la RAG. Trois stratégies principales :

| Stratégie | Avantages | Inconvénients |
|-----------|-----------|---------------|
| **Fixe** | Simple, prévisible | Coupe au milieu des phrases |
| **Sémantique** | Respecte le sens | Plus complexe, variable |
| **Récursif** | Équilibre taille/sens | Nécessite des délimiteurs |

### Préparation du DataFrame de chunks

Nous créons maintenant notre **base de connaissances** : un DataFrame pandas contenant tous les chunks avec leurs métadonnées.

**Structure du DataFrame** :
- `chunk_id` : Identifiant unique (0-50)
- `source` : Source du document (pour traçabilité)
- `text` : Contenu textuel du chunk
- `embedding` : Vecteur de 3072 dimensions (ajouté à l'étape suivante)

Cette structure permet de facilement filtrer, rechercher et enrichir les chunks avec des métadonnées supplémentaires (date, auteur, catégorie, etc.).

In [5]:
def chunk_fixed(text: str, chunk_size: int = 400, overlap: int = 50) -> List[str]:
    """
    Chunking fixe par nombre de mots.
    
    Args:
        text: Texte à découper
        chunk_size: Nombre de mots par chunk
        overlap: Chevauchement entre chunks (évite les coupures brutales)
    """
    words = text.split()
    chunks = []
    start = 0
    
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += (chunk_size - overlap)
    
    return chunks


def chunk_semantic(text: str, max_sentences: int = 5) -> List[str]:
    """
    Chunking sémantique par phrases.
    Regroupe les phrases en chunks cohérents.
    """
    import re
    # Découpage par phrases (approximatif)
    sentences = re.split(r'(?<=[.!?])\s+', text)
    
    chunks = []
    current_chunk = []
    
    for sentence in sentences:
        current_chunk.append(sentence)
        if len(current_chunk) >= max_sentences:
            chunks.append(" ".join(current_chunk))
            current_chunk = []
    
    # Dernier chunk
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    
    return chunks


def chunk_recursive(text: str, max_size: int = 500, delimiters: List[str] = None) -> List[str]:
    """
    Chunking récursif avec délimiteurs hiérarchiques.
    Essaie de découper par paragraphes, puis phrases, puis mots.
    """
    if delimiters is None:
        delimiters = ["\n\n", "\n", ". ", " "]
    
    if len(text) <= max_size or not delimiters:
        return [text] if text.strip() else []
    
    delimiter = delimiters[0]
    parts = text.split(delimiter)
    
    chunks = []
    current = ""
    
    for part in parts:
        if len(current) + len(part) + len(delimiter) <= max_size:
            current += (delimiter if current else "") + part
        else:
            if current:
                chunks.append(current)
            # Récursion avec délimiteur suivant si le part est trop grand
            if len(part) > max_size:
                chunks.extend(chunk_recursive(part, max_size, delimiters[1:]))
            else:
                current = part
    
    if current:
        chunks.append(current)
    
    return chunks


# Comparaison des stratégies
print("=== Comparaison des stratégies de chunking ===")
print(f"\nTexte source: {len(debate_text)} caractères")

chunks_fixed = chunk_fixed(debate_text, chunk_size=100, overlap=20)
chunks_semantic = chunk_semantic(debate_text, max_sentences=3)
chunks_recursive = chunk_recursive(debate_text, max_size=500)

print(f"\nChunking fixe (100 mots, overlap 20): {len(chunks_fixed)} chunks")
print(f"Chunking sémantique (3 phrases): {len(chunks_semantic)} chunks")
print(f"Chunking récursif (max 500 chars): {len(chunks_recursive)} chunks")

=== Comparaison des stratégies de chunking ===

Texte source: 98529 caractères

Chunking fixe (100 mots, overlap 20): 222 chunks
Chunking sémantique (3 phrases): 214 chunks
Chunking récursif (max 500 chars): 327 chunks


### Exemple guidé 1 - Stratégie de chunking hybride (crédité #21046)

Le chunking fixe coupe au milieu des phrases, le sémantique produit des chunks de taille variable, et le récursif necessite de bons delimiteurs. Une stratégie hybride combine les avantages de plusieurs approches.

**Objectif** : Implementer une fonction `chunk_hybrid(text, target_size, tolerance, overlap)` qui decoupe un texte en chunks de taille cible tout en respectant les frontieres de phrases.

**Indices** :
- Le chunking hybride fonctionne en deux phases : decouper par phrases (comme le sémantique), puis regrouper les phrases tant que la taille reste dans `[target_size - tolerance, target_size + tolerance]`
- Utiliser `re.split(r'(?<=[.!?])\s+', text)` pour obtenir les phrases
- Pour chaque phrase, l'ajouter au chunk courant si `len(chunk) + len(phrase) <= target_size + tolerance`
- Ajouter un overlap : conserver les N derniers caractères du chunk précèdent comme debut du suivant
- `# Étape 1` : Decouper le texte en phrases avec `re.split()`
- `# Étape 2` : Regrouper les phrases en chunks tout en respectant `target_size + tolerance`
- `# Étape 3` : Ajouter l'overlap entre chunks consécutifs et retourner la liste
- `# Indice` : Verifier votre résultat avec `len(chunk_hybrid(debate_text, 400, 100, 50))` et comparer avec les 51 chunks du chunking fixe

> **Exemple guidé (crédité #21046).** La résolution ci-dessous est celle du groupe de la PR #21046, conservée telle quelle (démo locale simulée sur un texte court) : validation des paramètres, regroupement par phrases, overlap et découpage des phrases isolées trop longues. L'exercice 1bis qui la suit mesure, lui, le coût de cet overlap sur le texte réel du débat.

In [6]:
# Préparation G26 avec l'assistance de ChatGPT. Démo locale simulée.
import re
def chunk_hybrid(text, target_size=400, tolerance=100, overlap=50):
    """Phrases regroupées autour d'une cible en caractères.

    Le chevauchement est inclus dans le plafond target_size+tolerance. Une phrase
    trop longue est découpée pour respecter ce plafond. Le minimum de taille est
    un objectif souple, impossible à garantir pour le dernier fragment.
    """
    for value in (target_size, tolerance, overlap):
        if isinstance(value, bool) or not isinstance(value, int):
            raise ValueError('Tailles entières attendues')
    if target_size <= 0 or tolerance < 0 or not 0 <= overlap < target_size:
        raise ValueError('Paramètres de découpage invalides')
    if not isinstance(text, str):
        raise ValueError('Texte attendu')
    maximum = target_size + tolerance
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s.strip()]
    if not sentences:
        return []
    chunks, current, has_new = [], '', False
    for sentence in sentences:
        remaining = sentence
        while remaining:
            separator = ' ' if current else ''
            if len(current) + len(separator) + len(remaining) <= maximum:
                current += separator + remaining
                has_new = True
                remaining = ''
            elif has_new:
                chunks.append(current)
                current = current[-overlap:] if overlap else ''
                has_new = False
            else:
                # Phrase isolée trop longue : progression garantie à chaque tour.
                capacity = maximum - len(current) - len(separator)
                current += separator + remaining[:capacity]
                remaining = remaining[capacity:]
                has_new = True
    if has_new:
        chunks.append(current)
    return chunks


_demo_text='Une phrase courte. Une deuxième phrase. Une troisième idée ! Une fin.'
_demo_chunks=chunk_hybrid(_demo_text,35,10,5)
for _i,_c in enumerate(_demo_chunks,1): print(_i,len(_c),repr(_c))


1 39 'Une phrase courte. Une deuxième phrase.'
2 35 'rase. Une troisième idée ! Une fin.'


### Exercice 1bis - Le prix de l'overlap : redondance du chunking hybride

L'exemple guidé 1 démontre `chunk_hybrid` sur un texte court de quatre phrases. Sur le texte réel du débat, l'overlap a un coût : chaque caractère repris du chunk précédent est stocké et transmis **deux fois** à l'index vectoriel.

**Objectif** : mesurer ce coût sur `debate_text`, en quantifiant la redondance réellement introduite par l'overlap.

**Indices** :
- `# Étape 1` : découper `debate_text` avec `chunk_hybrid(debate_text, 400, 100, 50)` et relever `nb_chunks_hybride`, puis `longueur_min` et `longueur_max` des chunks obtenus
- `# Étape 2` : pour chaque paire de chunks consécutifs, mesurer le chevauchement effectif — le plus long suffixe de `chunks_hybrides[i-1]` qui est aussi un préfixe de `chunks_hybrides[i]` (tester les longueurs décroissantes avec `endswith`/`startswith`)
- `# Étape 3` : sommer ces chevauchements en `overlap_total`, puis calculer `fraction_redondante = overlap_total / somme des longueurs de tous les chunks`
- `# Indice` : comparer `nb_chunks_hybride` aux 51 chunks du chunking fixe (l'indice de l'exemple guidé) ; attention, le chevauchement effectif peut être **inférieur** au paramètre `overlap` quand un nouveau chunk commence par une phrase fraîche

In [7]:
# Exercice 1bis - Le prix de l'overlap : redondance du chunking hybride

# Étape 1 : découpage et statistiques de longueur
chunks_hybrides = chunk_hybrid(debate_text, 400, 100, 50)
longueurs = [len(c) for c in chunks_hybrides]
nb_chunks_hybride = len(chunks_hybrides)
longueur_min = min(longueurs)
longueur_max = max(longueurs)

# Étape 2 : chevauchement effectif entre chunks consécutifs
def chevauchement_effectif(prev, curr):
    for k in range(min(len(prev), len(curr)), 0, -1):
        if prev.endswith(curr[:k]):
            return k
    return 0

chevauchements = [
    chevauchement_effectif(chunks_hybrides[i - 1], chunks_hybrides[i])
    for i in range(1, nb_chunks_hybride)
]

# Étape 3 : redondance totale
overlap_total = sum(chevauchements)
fraction_redondante = overlap_total / sum(longueurs)

print(f'Chunks hybrides : {nb_chunks_hybride} (vs 51 en chunking fixe)')
print(f'Longueur min / max : {longueur_min} / {longueur_max}')
print(f'Overlap total : {overlap_total} caractères')
print(f'Fraction redondante : {fraction_redondante:.1%}')

Chunks hybrides : 281 (vs 51 en chunking fixe)
Longueur min / max : 78 / 500
Overlap total : 14000 caractères
Fraction redondante : 12.4%


Les trois stratégies de chunking étant implementées, cette cellule compare leurs résultats sur le même texte source. Les différences de granularité entre chunking fixe, sémantique et récursif seront visibles dans les comptes de chunks produits.

In [8]:
# Visualisation des chunks
print("=== Exemple chunk fixe ===")
print(chunks_fixed[0][:300] + "..." if len(chunks_fixed[0]) > 300 else chunks_fixed[0])

print("\n=== Exemple chunk sémantique ===")
print(chunks_semantic[0][:300] + "..." if len(chunks_semantic[0]) > 300 else chunks_semantic[0])

print("\n=== Exemple chunk récursif ===")
print(chunks_recursive[0][:300] + "..." if len(chunks_recursive[0]) > 300 else chunks_recursive[0])

=== Exemple chunk fixe ===
First Debate: Ottawa, Illinois August 21, 1858 It was dry and dusty, between 10,000 and 12,000 people were in attendance when the debate began at 2:00 p.m. There were no seats or bleachers. Douglas charged Lincoln with trying to “abolitionize” the Whig and Democratic Parties. He also charged Lincoln...

=== Exemple chunk sémantique ===
First Debate: Ottawa, Illinois
August 21, 1858
It was dry and dusty, between 10,000 and 12,000 people were in attendance when the debate began at 2:00 p.m. There were no seats or bleachers. Douglas charged Lincoln with trying to “abolitionize” the Whig and Democratic Parties.

=== Exemple chunk récursif ===
First Debate: Ottawa, Illinois
August 21, 1858
It was dry and dusty, between 10,000 and 12,000 people were in attendance when the debate began at 2:00 p.m. There were no seats or bleachers.


### Analyse des résultats de chunking

Observons les différences obtenues :

| Stratégie | Nombre de chunks | Observations |
|-----------|------------------|--------------|
| **Fixe (100 mots, overlap 20)** | 222 | Nombreux chunks petits, overlap assure la continuité |
| **Sémantique (3 phrases)** | 214 | Chunks de taille variable, respecte le sens |
| **Récursif (max 500 chars)** | 327 | Plus de petits chunks, découpage hiérarchique |

**Points clés** :

1. **Chunking fixe** : Le premier exemple montre une coupure au milieu d'une phrase ("...He also charged Lincoln..."), typique de cette méthode.

2. **Chunking sémantique** : Le découpage respecte les limites de phrases, plus naturel pour la compréhension.

3. **Chunking récursif** : Découpe d'abord par paragraphes (`\n\n`), puis par phrases si trop long. Résultat : chunks courts mais cohérents.

**Pour la suite**, nous utilisons le chunking fixe avec **400 mots et overlap 50** - un bon compromis entre taille et continuité pour ce type de document narratif.

## 2.1 Choix de la stratégie

**Recommandations** :

- **Documents structurés** (code, JSON, markdown) → Chunking récursif
- **Texte narratif** (articles, livres) → Chunking sémantique
- **Données tabulaires** → Chunking fixe avec métadonnées

Pour la suite, nous utilisons le chunking fixe avec overlap (400 mots, overlap 50).

### Comprendre les embeddings

Un **embedding** est une représentation vectorielle d'un texte dans un espace à haute dimension. Chaque dimension capture un aspect sémantique du texte. **Sentence-BERT** (Reimers & Gurevych, 2019, arXiv:1908.10084) a rendu les embeddings de phrases efficaces pour la similarité sémantique via un fine-tuning siamese ; **DPR** (Karpukhin et al., 2020, arXiv:2004.04906) montre qu'un retriever dense bi-encoder surpasse BM25 sur les questions ouvertes.

**Exemple simplifié (2D)** :
```
"chat"      → [0.8, 0.2]  (animal, domestique)
"chien"     → [0.7, 0.3]  (animal, domestique)
"ordinateur" → [0.1, 0.9]  (objet, technologie)
```

En réalité, `text-embedding-3-large` génère des vecteurs à **3072 dimensions** pour capturer les nuances du langage.

**Propriétés clés** :

1. **Similarité sémantique** : Textes similaires ont des vecteurs proches (distance cosinus faible)
2. **Invariance** : Même sens avec différents mots → vecteurs similaires
3. **Efficacité** : Comparer 3072 nombres est plus rapide que comparer du texte brut

**Batch vs Single** : La fonction `create_embeddings_batch()` est **5-10x plus rapide** que des appels individuels pour de gros volumes. OpenAI autorise jusqu'à 2048 textes par batch.

**Résultat** : 51 vecteurs de 3072 dimensions, prêts pour la recherche vectorielle.

### Implémentation de la VectorStore

La classe `VectorStore` encapsule la logique de recherche vectorielle. Elle utilise scikit-learn pour le prototypage, mais en production vous utiliseriez une vraie base vectorielle.

**Points clés de l'implémentation** :

1. **Métrique cosinus** : `metric="cosine"` mesure l'angle entre vecteurs, pas la distance euclidienne
2. **Validation des embeddings** : Filtre les None pour éviter les erreurs
3. **Retour structuré** : Dicts avec `chunk_id`, `text`, `source`, et `score` pour traçabilité complète

In [9]:
# Création du DataFrame de chunks pour la suite
chunks = chunk_fixed(debate_text, chunk_size=400, overlap=50)

df_chunks = pd.DataFrame({
    "chunk_id": range(len(chunks)),
    "source": "Lincoln-Douglas Debate 1 (Ottawa, 1858)",
    "text": chunks
})

print(f"Base de connaissances: {len(df_chunks)} chunks")
df_chunks.head()

Base de connaissances: 51 chunks


,chunk_id,source,text
0,0,"Lincoln-Douglas Debate 1 (Ottawa, 1858)","First Debate: Ottawa, Illinois August 21, 1858..."
1,1,"Lincoln-Douglas Debate 1 (Ottawa, 1858)",were proclaimed wherever the Constitution rule...
2,2,"Lincoln-Douglas Debate 1 (Ottawa, 1858)",the Whig party and the Democratic party both s...
3,3,"Lincoln-Douglas Debate 1 (Ottawa, 1858)",name and disguise of a Republican party. (Laug...
4,4,"Lincoln-Douglas Debate 1 (Ottawa, 1858)",with such views as the circumstances and exige...


### Analyse de la base de connaissances

Le DataFrame créé contient notre base de connaissances structurée, prête pour la vectorisation.

**Structure du DataFrame** :

| Colonne | Type | Description |
|---------|------|-------------|
| `chunk_id` | int | Identifiant unique (0-50) |
| `source` | str | Source du document (traçabilité) |
| `text` | str | Contenu textuel du chunk |

**Métriques observées** :

- **Nombre de chunks** : 51
- **Stratégie** : Chunking fixe avec 400 mots et overlap 50
- **Résultat** : Cohérent avec la taille du document (~98k caractères)

**Vérification qualité (via `head()`)** :

Les 5 premiers chunks montrent :
1. **Chunk 0** : Début du débat ("First Debate: Ottawa, Illinois August 21, 1858...")
2. **Chunks suivants** : Continuité narrative avec overlap
3. **Source cohérente** : Tous les chunks référencent la même source

**Calcul de l'overlap** :

```
Chunk 0: mots 0-400
Chunk 1: mots 350-750  (50 mots de chevauchement avec chunk 0)
Chunk 2: mots 700-1100 (50 mots de chevauchement avec chunk 1)
...
```

**Avantages de l'overlap** :
- Évite les coupures brutales de contexte
- Améliore la récupération de phrases à cheval sur deux chunks
- Coût : ~12.5% de redondance (50/400)

**Prochaine étape** : Générer les embeddings pour chaque chunk avec `text-embedding-3-large`.

***

## Partie 3 : Embeddings et Recherche Vectorielle

### 3.1 Génération des embeddings

OpenAI propose plusieurs modèles d'embeddings :

| Modèle | Dimensions | Performance | Coût |
|--------|------------|-------------|------|
| `text-embedding-3-small` | 1536 | Bon | $ |
| `text-embedding-3-large` | 3072 | Excellent | $$ |
| `text-embedding-ada-002` | 1536 | Bon (legacy) | $ |

In [10]:
def create_embedding(text: str, model: str = "text-embedding-3-large") -> List[float]:
    """
    Génère un embedding pour un texte donné.
    
    Args:
        text: Texte à vectoriser
        model: Modèle d'embedding OpenAI
    
    Returns:
        Vecteur d'embedding (liste de floats)
    """
    try:
        response = client.embeddings.create(
            model=model,
            input=[text]
        )
        return response.data[0].embedding
    except Exception as e:
        print(f"Erreur embedding: {e}")
        return None


def create_embeddings_batch(texts: List[str], model: str = "text-embedding-3-large") -> List[List[float]]:
    """
    Génère des embeddings en batch (plus efficace).
    
    Note: OpenAI supporte jusqu'à 2048 textes par requête.
    """
    try:
        response = client.embeddings.create(
            model=model,
            input=texts
        )
        return [item.embedding for item in response.data]
    except Exception as e:
        print(f"Erreur batch embedding: {e}")
        return [None] * len(texts)


# Génération des embeddings pour tous les chunks
print("Génération des embeddings...")
embeddings = create_embeddings_batch(df_chunks["text"].tolist())
df_chunks["embedding"] = embeddings

print(f"Embeddings générés: {len([e for e in embeddings if e])} / {len(embeddings)}")
print(f"Dimension des vecteurs: {len(embeddings[0]) if embeddings[0] else 'N/A'}")

Génération des embeddings...


Embeddings générés: 51 / 51
Dimension des vecteurs: 3072


### Analyse des embeddings générés

Les résultats confirment une vectorisation réussie de notre base de connaissances.

**Métriques observées** :

| Métrique | Valeur | Signification |
|----------|--------|---------------|
| **Embeddings générés** | 51/51 | 100% de succès, aucune erreur API |
| **Dimension des vecteurs** | 3072 | `text-embedding-3-large` (haute précision) |
| **Chunks indexés** | 51 | Base de connaissances complète |

**Implications pour la recherche** :

1. **Espace vectoriel** : Chaque chunk est maintenant un point dans un espace à 3072 dimensions
2. **Distance sémantique** : Des chunks parlant du même concept seront proches dans cet espace
3. **Efficacité** : La recherche k-NN sera rapide même avec des milliers de chunks

**Comparaison des modèles d'embeddings** :

| Modèle | Dimensions | Taille base (51 chunks) | Performance recherche |
|--------|------------|------------------------|----------------------|
| `text-embedding-3-small` | 1536 | ~300 KB | Bonne |
| `text-embedding-3-large` | 3072 | ~600 KB | **Excellente** ✓ |
| `text-embedding-ada-002` | 1536 | ~300 KB | Bonne (legacy) |

**Note technique** : L'utilisation de `create_embeddings_batch()` au lieu d'appels individuels réduit le temps d'exécution de **~10 secondes à ~2 secondes** pour 51 chunks. En production avec des milliers de documents, cette optimisation est critique.

**Prochaine étape** : Initialiser le VectorStore avec ces embeddings pour la recherche k-NN.

### Anatomie d'une réponse RAG

Le pipeline `rag_query()` exécute 4 étapes :

**1. Retrieval (Récupération)** :
```python
query_embedding = create_embedding(question)
chunks = vector_store.search(query_embedding, k=3)
```
Convertit la question en vecteur, puis cherche les 3 chunks les plus proches.

**2. Construction du contexte** :
```python
context = "\n\n".join([f"[Source: {c['source']}]\n{c['text']}" for c in chunks])
```
Assemble les chunks récupérés en un seul texte avec métadonnées.

**3. Prompt augmenté** :
```python
system_prompt = "Réponds UNIQUEMENT basé sur le contexte fourni..."
user_prompt = f"Contexte:\n{context}\n\nQuestion: {question}"
```
Injecte le contexte dans le prompt pour "augmenter" les connaissances du modèle.

**4. Génération** :
```python
response = client.chat.completions.create(...)
```
Le LLM génère une réponse basée sur le contexte fourni, pas sur ses connaissances pré-entraînées.

**Paramètrès clés** :
- `température` : Non supporté par gpt-5-mini (utilise la valeur par défaut 1.0)
- `max_completion_tokens=2000` : Budget de génération. Pour un modèle de raisonnement
  comme gpt-5-mini, ce budget doit être suffisamment large car les tokens de
  raisonnement sont déduits du même budget (un budget trop bas, ex: 500, pouvait
  être entièrement consommé par le raisonnement sans texte final visible — issue #3571).

**Résultat attendu** : Une réponse précise citant les chunks utilisés, avec traçabilité complète via les métadonnées de tokens.

## 3.2 Recherche k-NN (k-Nearest Neighbors)

Pour la recherche, on calcule la distance entre l'embedding de la question et ceux des chunks.

## Importance des citations en RAG

Les citations sont **essentielles** pour :

1. **Vérifiabilité** : L'utilisateur peut vérifier la source de l'information
2. **Confiance** : Une réponse avec sources est plus crédible qu'une affirmation sans preuve
3. **Debugging** : Identifier si le problème vient du retrieval (mauvais chunks) ou de la génération (mauvaise interprétation)
4. **Conformité** : Certains domaines (médical, juridique) exigent la traçabilité complète

**Anti-pattern** : Générer une réponse sans indiquer les sources utilisées. Cela empêche la vérification et rend le système opaque.

### Responses API vs Chat Completions

La **Responses API** fournit un état de conversation côté serveur, tandis que Chat Completions demande à l'application de gérer l'historique.

**Différences techniques** :

| Aspect | Chat Completions | Responses API |
|--------|------------------|---------------|
| **Historique** | Géré et renvoyé par l'application | Chaîné par `previous_response_id` |
| **Prompt caching** | Disponible sous ses propres conditions | Disponible sous les mêmes conditions de cache fournisseur |
| **Mesure des tokens** | Champs `usage` de la réponse | Champs `usage` de la réponse |
| **Persistance** | Côté application | Côté serveur avec `store=True` |

`previous_response_id` simplifie le transport de l'historique, mais ne rend pas cet historique gratuit : les tokens d'entrée historiques restent comptabilisés. Il ne démontre pas non plus, à lui seul, un hit de prompt cache. La cellule de mesure plus bas affiche donc les champs `usage` réellement renvoyés pour chaque tour, sans annoncer un pourcentage d'économie théorique. Les appels d'embeddings sont suivis séparément du coût de génération.

**Fallback implémenté** : si la Responses API n'est pas disponible, le code bascule sur Chat Completions. Ce fallback assure la compatibilité de l'appel courant, mais ne reprend pas automatiquement le même état de conversation via `previous_response_id` ; l'application devrait reconstruire explicitement l'historique pour conserver cette continuité.

### Transition vers les bonnes pratiques

Nous avons maintenant un pipeline RAG complet fonctionnel. Cependant, pour passer en **production**, plusieurs optimisations sont nécessaires.

**Prochaines étapes** :

1. **Monitoring et métriques** : Mesurer la qualité (relevance, precision, recall)
2. **Scaling** : Base vectorielle distribuée (Pinecone, Qdrant)
3. **Optimisation coûts** : Cache intelligent, batch processing
4. **Évaluation continue** : Tests A/B sur différentes stratégies de chunking

La section suivante couvre ces aspects avec des recommandations concrètes.

In [11]:
from sklearn.neighbors import NearestNeighbors

class VectorStore:
    """
    Base vectorielle simple basée sur scikit-learn.
    En production, utiliser Pinecone, Qdrant, Weaviate, ou Chroma.
    """
    
    def __init__(self, df: pd.DataFrame, embedding_col: str = "embedding"):
        self.df = df
        self.embedding_col = embedding_col
        
        # Filtrer les embeddings valides
        valid_mask = df[embedding_col].apply(lambda x: x is not None)
        self.valid_indices = df[valid_mask].index.tolist()
        
        # Construire la matrice de vecteurs
        vectors = np.array(df.loc[self.valid_indices, embedding_col].tolist())
        
        # Index k-NN
        self.nn = NearestNeighbors(n_neighbors=min(5, len(vectors)), metric="cosine")
        self.nn.fit(vectors)
        
        print(f"VectorStore initialisé: {len(self.valid_indices)} vecteurs")
    
    def search(self, query_embedding: List[float], k: int = 3) -> List[Dict[str, Any]]:
        """
        Recherche les k chunks les plus similaires.
        
        Returns:
            Liste de dicts avec chunk_id, text, source, score
        """
        distances, indices = self.nn.kneighbors([query_embedding], n_neighbors=k)
        
        results = []
        for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
            original_idx = self.valid_indices[idx]
            row = self.df.iloc[original_idx]
            results.append({
                "chunk_id": row["chunk_id"],
                "text": row["text"],
                "source": row["source"],
                "score": 1 - dist  # Cosine similarity (1 = identique)
            })
        
        return results


# Initialisation du VectorStore
vector_store = VectorStore(df_chunks)

VectorStore initialisé: 51 vecteurs


### Exemple guidé 2 - Recherche vectorielle avec seuil de confiance (crédité #21046)

Le VectorStore utilise k-NN pour trouver les chunks les plus proches, mais ne filtre pas les résultats peu pertinents. Dans un système de production, il est essentiel de ne retourner que les chunks dont la similarite depasse un seuil.

**Objectif** : Implementer une fonction `search_with_confidence(vector_store, question, k, min_score)` qui effectue une recherche vectorielle et retourne uniquement les chunks dont le score depasse un seuil, avec un niveau de confiance global.

**Indices** :
- Reutiliser `create_embedding()` pour vectoriser la question, puis `vector_store.search()` pour la recherche
- Filtrer les résultats avec `min_score` et calculer un niveau de confiance : "haute" si meilleur score > 0.7, "moyenne" si > 0.5, "basse" sinon
- Retourner un dict `{"results": List[Dict], "confidence": str, "total_found": int, "kept": int}`
- `# Étape 1` : Créer l'embedding de la question avec `create_embedding()`
- `# Étape 2` : Appeler `vector_store.search(query_embedding, k=k)` et filtrer par `min_score`
- `# Étape 3` : Determiner le niveau de confiance a partir du meilleur score des résultats filtrès
- `# Indice` : Si aucun résultat ne depasse le seuil, retourner confiance "basse" avec liste vide

> **Exemple guidé (crédité #21046).** La résolution ci-dessous est celle du groupe de la PR #21046, conservée telle quelle (démo sur un magasin simulé aux scores contrôlés) : validation de `k` et du seuil cosinus, filtration des résultats et niveau de confiance dérivé du meilleur score gardé. L'exercice 2bis qui la suit étudie, lui, la sensibilité du niveau de confiance au choix du seuil.

In [12]:
# Préparation G26 avec l'assistance de ChatGPT. Démo locale simulée.
import math
def search_with_confidence(vector_store, question, k=5, min_score=0.5, embedding_fn=None):
    """Les niveaux de confiance sont les heuristiques de l'énoncé, pas des probabilités."""
    if isinstance(k, bool) or not isinstance(k, int) or k < 1:
        raise ValueError('k doit être un entier strictement positif')
    if isinstance(min_score, bool) or not isinstance(min_score, (int,float)) or not math.isfinite(min_score) or not -1 <= min_score <= 1:
        raise ValueError('Seuil cosinus invalide')
    embedder = embedding_fn or globals().get('create_embedding')
    if embedder is None:
        raise RuntimeError('Définir create_embedding ou fournir embedding_fn')
    embedding = embedder(question)
    if embedding is None:
        raise RuntimeError('Échec de vectorisation : aucune recherche effectuée')
    found = vector_store.search(embedding, k=k)
    kept = [r for r in found if r['score'] >= min_score]
    best = max((r['score'] for r in kept), default=-1)
    confidence = 'haute' if best > 0.7 else 'moyenne' if best > 0.5 else 'basse'
    return {'results':kept, 'confidence':confidence, 'total_found':len(found), 'kept':len(kept)}


class _DemoStore:
    def search(self, query_embedding, k=5):
        return [{'score':s, 'text':f'extrait {i}'} for i,s in enumerate([0.82,0.61,0.20])][:k]
print('Simulation du moteur :',search_with_confidence(_DemoStore(),'question',embedding_fn=lambda q:[1,0]))
# Pour le moteur réel du notebook : search_with_confidence(vector_store, question)


Simulation du moteur : {'results': [{'score': 0.82, 'text': 'extrait 0'}, {'score': 0.61, 'text': 'extrait 1'}], 'confidence': 'haute', 'total_found': 3, 'kept': 2}


### Exercice 2bis - Calibrer le seuil : frontières des niveaux de confiance

L'exemple guidé 2 démontre `search_with_confidence` sur un point unique (meilleur score 0.82, confiance "haute"). En production, le choix de `min_score` décide **quels résultats survivent** — et le niveau de confiance affiché ne dépend plus que du meilleur score gardé.

**Objectif** : balayer le seuil sur un magasin aux scores contrôlés et dresser la carte des transitions de confiance.

**Indices** :
- `# Étape 1` : construire un magasin de démonstration (sur le modèle du `_DemoStore` de l'exemple guidé) dont `search()` renvoie les scores contrôlés `[0.95, 0.75, 0.62, 0.51, 0.34, 0.12]`
- `# Étape 2` : balayer `min_score` sur `[0.0, 0.3, 0.5, 0.55, 0.65, 0.8]` et relever, pour chaque valeur, le couple `(kept, confidence)` retourné par `search_with_confidence`
- `# Étape 3` : identifier `frontieres_confiance` — les valeurs critiques de meilleur score gardé où le niveau change (le seuil "haute" est à 0.7, "moyenne" à 0.5) — et dire entre quels scores contrôlés consécutifs chaque frontière tombe
- `# Indice` : la confiance ne dépend QUE du meilleur score gardé ; observez ce qui se passe quand `min_score` exclut le meilleur (0.95) mais garde les suivants — la confiance peut **tomber** alors que le seuil monte

In [13]:
# Exercice 2bis - Calibrer le seuil : frontieres des niveaux de confiance
# TODO etudiant : etudier la sensibilite du niveau de confiance au choix du seuil
# Etape 1 : construire un magasin de demonstration avec les scores controles
#           scores_controles = [0.95, 0.75, 0.62, 0.51, 0.34, 0.12]
# Etape 2 : balayer min_score sur [0.0, 0.3, 0.5, 0.55, 0.65, 0.8] et relever
#           (kept, confidence) pour chaque valeur dans table_calibration
# Etape 3 : identifier frontieres_confiance, les valeurs critiques de meilleur
#           score garde ou le niveau change (haute > 0.7, moyenne > 0.5)
# Indice : quand min_score exclut le meilleur score, la confiance peut tomber
scores_controles = [0.95, 0.75, 0.62, 0.51, 0.34, 0.12]
seuils_balayes = None  # TODO etudiant
table_calibration = None  # TODO etudiant
frontieres_confiance = None  # TODO etudiant
print("Exercice 2bis a completer : calibration du seuil de confiance")

Exercice 2bis a completer : calibration du seuil de confiance


Le VectorStore est maintenant initialisé avec les chunks et leurs embeddings. Cette cellule effectue une première requête de recherche sémantique pour valider que la récupération fonctionne correctement avant de l'intégrer dans le pipeline RAG complet.

In [14]:
# Test de recherche
question = "What did Lincoln argue about slavery?"

# Embedding de la question
query_embedding = create_embedding(question)

# Recherche des chunks pertinents
results = vector_store.search(query_embedding, k=3)

print(f"Question: {question}\n")
print("=== Chunks retrouvés ===")
for i, r in enumerate(results):
    print(f"\n[{i+1}] Score: {r['score']:.3f} | Source: {r['source']}")
    print(f"    {r['text'][:400]}...")

Question: What did Lincoln argue about slavery?

=== Chunks retrouvés ===

[1] Score: 0.628 | Source: Lincoln-Douglas Debate 1 (Ottawa, 1858)
    proclaims his Abolition doctrines. Let me read a part of them. In his speech at Springfield to the Convention, which nominated him for the Senate, he said: "In my opinion it will not cease until a crisis shall have been reached and passed. 'A house divided against itself cannot stand.' I believe this government cannot endure permanently half Slave and half Free . I do not expect the Union to be di...

[2] Score: 0.579 | Source: Lincoln-Douglas Debate 1 (Ottawa, 1858)
    them not to have it if they do not want it. [Applause and laughter.] I do not mean that if this vast concourse of people were in a Territory of the United States, any one of them would be obliged to have a slave if he did not want one; but I do say that, as I understand the Dred Scott decision, if any one man wants slaves, all the rest have no way of keeping that one man from

### Interprétation des scores de similarité

Les résultats de recherche montrent :

| Rang | Score | Interprétation |
|------|-------|----------------|
| 1 | 0.628 | **Bonne correspondance** - Le chunk contient directement la doctrine d'abolition de Lincoln |
| 2 | 0.579 | **Pertinence moyenne** - Discussion sur les droits territoriaux liés à l'esclavage |
| 3 | 0.572 | **Pertinence moyenne** - Contexte historique sur les principes des pères fondateurs |

**Analyse du score** :

- **Score > 0.7** : Excellent match, très pertinent
- **0.5 < Score < 0.7** : Pertinent, mais contexte indirect
- **Score < 0.5** : Faible pertinence, risque de bruit

Ici, le **chunk #1 avec score 0.628** est le plus pertinent. Il mentionne explicitement les "Abolition doctrines" de Lincoln dans son discours à Springfield, répondant directement à la question "What did Lincoln argue about slavery?".

**Point important** : Même un score de 0.628 (62.8% de similarité) est considéré comme **bon** en RAG. Les scores parfaits (>0.9) ne s'obtiennent que pour des textes quasi-identiques.

**Distance cosinus** : Le score affiché est `1 - distance`, donc 1 = identique, 0 = orthogonal (aucune similarité).

***

## Partie 4 : RAG avec Chat Completions

### 4.1 Pipeline RAG classique

### Qualité des citations : critères d'évaluation

Une bonne réponse RAG avec citations doit respecter plusieurs critères :

**1. Précision des références** :
- Chaque affirmation doit être associée à une source [1], [2], etc.
- Les citations multiples sont encouragées si plusieurs sources confirment la même information

**2. Format structuré** :
```
RÉPONSE: Lincoln argued that... [1] while Douglas maintained... [2]
SOURCES UTILISÉES: [1, 2]
CONFIANCE: haute
```

**3. Niveaux de confiance** :
- **Haute** : Information présente dans plusieurs sources, cohérente
- **Moyenne** : Information dans une seule source, claire
- **Basse** : Information implicite, nécessite inférence

**Exemple de sortie attendue** :
```
RÉPONSE: Les points clés de désaccord étaient :
1. L'extension de l'esclavage [1, 2]
2. La souveraineté populaire vs restriction fédérale [1, 3]
3. L'interprétation de la Déclaration d'Indépendance [2]

SOURCES UTILISÉES: [1, 2, 3]
CONFIANCE: haute
```

**Vérification de qualité** : Comparer les extraits des chunks récupérés avec les citations dans la réponse pour s'assurer de la fidélité.

In [15]:
def rag_query(question: str, vector_store: VectorStore, k: int = 3) -> Dict[str, Any]:
    """
    Pipeline RAG complet avec Chat Completions.
    
    Args:
        question: Question de l'utilisateur
        vector_store: Base vectorielle
        k: Nombre de chunks à récupérer
    
    Returns:
        Dict avec réponse, sources, et métadonnées
    """
    # 1. Retrieval
    query_embedding = create_embedding(question)
    chunks = vector_store.search(query_embedding, k=k)
    
    # 2. Construction du contexte
    context = "\n\n".join([
        f"[Source: {c['source']} | Chunk {c['chunk_id']}]\n{c['text']}"
        for c in chunks
    ])
    
    # 3. Prompt augmenté
    system_prompt = """Tu es un assistant expert en histoire américaine.
Réponds aux questions en te basant UNIQUEMENT sur le contexte fourni.
Si l'information n'est pas dans le contexte, dis-le clairement.
Cite tes sources avec le format [Chunk X]."""
    
    user_prompt = f"""Contexte:
{context}

Question: {question}

Réponds de manière précise en citant les sources."""
    
    # 4. Génération
    # Note: temperature not supported by gpt-5-mini, uses default (1.0)
    # Budget relevé à 2000 : gpt-5-mini est un modèle de raisonnement dont les
    # tokens de raisonnement sont déduits de max_completion_tokens. Avec 500,
    # tout le budget pouvait être consommé par le raisonnement sans texte final
    # visible (issue #3571). 2000 laisse une marge confortable pour le contenu.
    response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_completion_tokens=2000
    )
    
    return {
        "question": question,
        "answer": response.choices[0].message.content,
        "sources": chunks,
        "model": DEFAULT_MODEL,
        "tokens": {
            "prompt": response.usage.prompt_tokens,
            "completion": response.usage.completion_tokens
        }
    }


# Test du pipeline RAG
result = rag_query(
    "What were Lincoln's main arguments about slavery in the debate?",
    vector_store
)

print("=== Réponse RAG ===")
print(f"\nQuestion: {result['question']}")
print(f"\nRéponse:\n{result['answer']}")
print(f"\nTokens utilisés: {result['tokens']}")

=== Réponse RAG ===

Question: What were Lincoln's main arguments about slavery in the debate?

Réponse:
D’après les extraits fournis, les principaux arguments de Lincoln lors du débat sont les suivants :

- La thèse du « house divided » : Lincoln affirmait que « this government cannot endure permanently half Slave and half Free » et que « It will become all one thing, or all the other » ; il soutenait qu’il fallait « arrest the further spread of [slavery], and place it where the public mind shall rest in the belief that it is in the course of ultimate extinction » [Chunk 9].  

- Il accusait Stephen Douglas de chercher à « nationalize » (rendre national) l’esclavage — c’est-à-dire d’étendre et d’affirmer la légalité de l’esclavage à l’échelle nationale — et porta cette charge pendant le débat [Chunk 0].  

- Il passa la séance en grande partie sur la défensive, niant les allégations de Douglas (notamment celles relatives à la convention et à la plateforme de 1854) ; le compte rendu in

### Analyse du pipeline RAG classique

L'exécution démontre le fonctionnement complet du pipeline RAG avec Chat Completions.

**Résultats observés** :

- La sortie précédente affiche directement les champs `prompt` et `completion` renvoyés par `response.usage` ; elle reste la source de vérité pour ce run.
- La réponse cite les chunks récupérés et distingue les principaux arguments attribués à Lincoln.
- La présence d'une citation ne prouve pas à elle seule que le chunk étaye exactement l'affirmation : cette fidélité se contrôle contre le texte source.
- Le modèle indique aussi lorsque des précisions demandées ne figurent pas dans le contexte fourni.

> **Note sur le budget `max_completion_tokens`** : `gpt-5-mini` est un modèle de
> raisonnement dont les tokens de raisonnement sont déduits du budget
> `max_completion_tokens`. Un budget trop bas peut être entièrement consommé sans
> laisser de texte final visible (issue #3571). Le pipeline utilise un budget de
> 2000 pour laisser une marge au contenu ; il s'agit d'une configuration, pas d'une
> mesure de consommation.

**Limitation identifiée** : le notebook mesure les tokens renvoyés par l'API, mais ne déduit pas un coût monétaire sans grille tarifaire versionnée. La section suivante vérifie séparément le chaînage multi-tour avec `previous_response_id`.

### Filtrage par seuil : écarter les résultats faibles

La recherche k-NN renvoie les `k` voisins les plus proches, même lorsque leurs scores sont faibles. La fonction présentée plus bas applique uniquement un **filtrage par seuil** au classement existant : elle ne recalcule aucun score et conserve l'ordre des survivants. Il ne s'agit donc pas d'un reranking.

**Principe** : conserver les chunks dont le score est supérieur ou égal à `min_score`.

| Exemple illustratif | Chunks k=5 | Après filtrage (`min_score=0.3`) | Résultat |
|---------------------|------------|----------------------------------|----------|
| Scores tous au-dessus du seuil | [0.8, 0.7, 0.6, 0.4, 0.3] | [0.8, 0.7, 0.6, 0.4, 0.3] | 5 chunks conservés |
| Plusieurs scores faibles | [0.4, 0.3, 0.2, 0.1, 0.05] | [0.4, 0.3] | 3 chunks éliminés |
| Scores intermédiaires | [0.5, 0.5, 0.4, 0.2, 0.1] | [0.5, 0.5, 0.4] | 2 chunks éliminés |

Ces valeurs illustrent le mécanisme, pas des seuils universels de confiance. Un score cosinus dépend du modèle d'embedding, du corpus, de la requête et de la distribution observée. `min_score` doit être calibré sur un jeu de questions annoté en mesurant notamment la précision et le rappel.

**Techniques distinctes et plus avancées** :
- **Reranking cross-encoder** : réévaluer les paires `(question, chunk)` avec un modèle spécialisé tel que `ms-marco-MiniLM-L-6-v2`, ce qui produit un nouveau score et potentiellement un nouvel ordre ;
- **Diversification** : limiter les chunks redondants, par exemple avec MMR ou clustering ;
- **Décroissance temporelle** : intégrer l'âge des documents lorsque le cas d'usage le justifie.

Le filtrage local n'ajoute aucun appel LLM, mais son intérêt doit être évalué sur la qualité du retrieval plutôt que déduit du seul score.

***

## Partie 5 : RAG avec Responses API (Moderne)

La **Responses API** (2025) fournit notamment un état de conversation côté serveur :

| Fonctionnalité | Rôle |
|----------------|------|
| `store=True` | Persistance des réponses côté serveur |
| `previous_response_id` | Chaînage multi-tour sans reconstruire manuellement la liste des messages |
| Champs `usage` | Mesure des tokens comptabilisés pour chaque appel |
| Outils intégrés | Web search, file search, code interpreter |

Le chaînage ne prouve ni un hit de prompt cache ni une réduction de tokens. Ces effets se vérifient uniquement avec les champs d'usage dédiés renvoyés par l'API.

In [16]:
def rag_query_responses_api(
    question: str,
    vector_store: VectorStore,
    previous_response_id: str = None,
    k: int = 3
) -> Dict[str, Any]:
    """
    Pipeline RAG avec Responses API.
    
    Avantages:
    - Conversations multi-turn avec previous_response_id
    - Persistence côté serveur
    """
    # 1. Retrieval
    query_embedding = create_embedding(question)
    chunks = vector_store.search(query_embedding, k=k)
    
    # 2. Construction du contexte (format texte simple pour Responses API)
    context = "\n\n".join([
        f"[Source: {c['source']} | Chunk {c['chunk_id']}]\n{c['text']}"
        for c in chunks
    ])
    
    # 3. Prompt complet
    full_prompt = f"""Tu es un assistant expert. Réponds en citant les sources [Chunk X].

Contexte:
{context}

Question: {question}"""
    
    # 4. Appel Responses API
    try:
        kwargs = {
            "model": DEFAULT_MODEL,
            "input": full_prompt,
            "store": True
        }
        if previous_response_id:
            kwargs["previous_response_id"] = previous_response_id
        
        response = client.responses.create(**kwargs)
        
        # Extraire le texte de la réponse
        answer = ""
        if response.output:
            for item in response.output:
                if hasattr(item, 'content'):
                    for content in item.content:
                        if hasattr(content, 'text'):
                            answer += content.text
        
        return {
            "question": question,
            "answer": answer if answer else "Pas de réponse",
            "response_id": response.id,  # Pour le chaînage
            "sources": chunks,
            "api": "responses",
            "usage": {
                "input_tokens": getattr(response.usage, "input_tokens", None),
                "output_tokens": getattr(response.usage, "output_tokens", None),
                "total_tokens": getattr(response.usage, "total_tokens", None),
            } if getattr(response, "usage", None) else None
        }
        
    except Exception as e:
        # Fallback sur Chat Completions si Responses API non disponible
        print(f"Note: Responses API non disponible. IMPORTANT: Le fallback Chat Completions NE REPREND PAS le même historique via previous_response_id ({e}), utilisation de Chat Completions")
        return rag_query(question, vector_store, k)


# Test Responses API
print("=== Test RAG avec Responses API ===")
result1 = rag_query_responses_api(
    "What did Lincoln say about slavery?",
    vector_store
)

print(f"\nQuestion 1: {result1['question']}")
print(f"Réponse: {result1['answer'][:1000]}...")
print(f"Response ID: {result1.get('response_id', 'N/A')}")

=== Test RAG avec Responses API ===



Question 1: What did Lincoln say about slavery?
Réponse: Lincoln a dit notamment qu’il s’oppose à l’extension de l’esclavage dans les territoires et qu’un « house divided against itself cannot stand » — le pays ne peut pas subsister « moitié Esclave, moitié Libre » indéfiniment ; il prédit qu’il cessera d’être divisé et deviendra « tout l’un ou tout l’autre » (c.-à-d. soit l’esclavage sera arrêté et mis en voie d’extinction, soit il se répandra partout). [Chunk 9]

Il affirmait aussi qu’on ne doit pas laisser l’esclavage entrer dans notre territoire libre (comparable, selon lui, au rétablissement de la traite africaine) et qu’il fallait empêcher son expansion dans des territoires comme le Nebraska. [Chunk 20]

En même temps, il déclara qu’il n’avait « no purpose, directly or indirectly, to interfere with the institution of slavery in the States where it exists » — il n’avait ni le droit légal ni l’intention d’abolir l’esclavage dans les États où il était établi — et il déclara ne pas 

La première requête Responses API ayant renvoyé un `response_id`, cette cellule teste le chaînage multi-tour : le contexte de la conversation précédente est automatiquement préservé sans nécessiter de renvoyer l'historique complet des messages.

In [17]:
# Multi-turn avec previous_response_id
print("=== Conversation multi-turn ===")

if result1.get('response_id'):
    # Question de suivi utilisant le contexte précédent
    result2 = rag_query_responses_api(
        "And what was Douglas's counter-argument?",
        vector_store,
        previous_response_id=result1['response_id']
    )
    
    print(f"\nQuestion 2 (suivi): {result2['question']}")
    print(f"Réponse: {result2['answer'][:500]}...")
    print("\nNote: Le contexte de la question 1 est automatiquement inclus via previous_response_id")
else:
    print("Responses API non disponible pour le multi-turn")

=== Conversation multi-turn ===



Question 2 (suivi): And what was Douglas's counter-argument?
Réponse: Douglas répliqua en plusieurs points principaux.

- Il invoqua la souveraineté populaire : la loi du Nebraska laissait les habitants libres de « former et régler leurs institutions domestiques » — c’est au peuple du territoire (puis de l’État) de décider pour ou contre l’esclavage. (Argument général sur la portée de la loi.) [Chunk 31]

- Quant à l’amendement proposé par M. Chase (autorisant expressément le peuple à exclure l’esclavage), Douglas dit qu’il ne l’avait pas rejeté par complaisance p...

Note: Le contexte de la question 1 est automatiquement inclus via previous_response_id


In [18]:
# Mesure réelle via les champs usage renvoyés par l'API
print("=== Mesure réelle des champs usage (2 tours) ===")
print("| Tour | API | input_tokens | output_tokens | total_tokens |")
print("|------|-----|--------------|---------------|--------------|")

res1 = globals().get("result1")
res2 = globals().get("result2")
for label, result in [("Q1", res1), ("Q2", res2)]:
    usage = (result or {}).get("usage") or {}
    api_name = (result or {}).get("api", "indisponible")
    print(
        f"| {label} | {api_name} | {usage.get('input_tokens')} | "
        f"{usage.get('output_tokens')} | {usage.get('total_tokens')} |"
    )

print(
    "Note : seules les valeurs lues dans response.usage sont affichées ; "
    "le coût des embeddings n'est pas estimé sans métrique d'usage fournie."
)

=== Mesure réelle des champs usage (2 tours) ===
| Tour | API | input_tokens | output_tokens | total_tokens |
|------|-----|--------------|---------------|--------------|
| Q1 | responses | 1531 | 887 | 2418 |
| Q2 | responses | 3368 | 1053 | 4421 |
Note : seules les valeurs lues dans response.usage sont affichées ; le coût des embeddings n'est pas estimé sans métrique d'usage fournie.


### Analyse du multi-turn avec Responses API

Cette exécution vérifie le chaînage de deux appels RAG avec `previous_response_id`. La table imprimée par la cellule précédente expose directement, pour Q1 et Q2, l'API utilisée ainsi que `input_tokens`, `output_tokens` et `total_tokens`. Cette sortie est l'unique source de vérité numérique : les valeurs ne sont pas recopiées dans cette cellule markdown, car elles peuvent varier d'une génération à l'autre.

**Ce que la sortie établit** :

1. Q1 renvoie un `response_id`, ensuite fourni comme `previous_response_id` à Q2.
2. La formulation anaphorique de Q2 produit une réponse centrée sur le contre-argument de Douglas.
3. Les valeurs affichées viennent directement de `response.usage`; elles ne sont ni estimées ni converties en coût.

**Ce que la sortie n'établit pas** :

- Elle ne démontre pas une économie de tokens : il faudrait comparer des scénarios équivalents et leurs champs d'usage détaillés.
- Elle ne sépare pas les éventuels tokens mis en cache ou refacturés ; cette information demande des champs d'usage dédiés et une grille tarifaire versionnée.
- Elle ne prouve pas à elle seule que le contexte est reproduit fidèlement : la réponse reste à contrôler contre les chunks cités.

Le bénéfice observé ici est donc la **continuité d'API et d'orchestration**, pas une réduction de coût mesurée.

## 5.1 Continuité du Multi-turn RAG

Avec `previous_response_id`, l'API conserve l'état de la réponse précédente côté serveur et permet de chaîner une question de suivi. Cela évite à l'application de reconstruire manuellement la liste des messages, mais les tokens historiques restent comptabilisés et ce notebook reconstruit encore un contexte RAG frais à chaque tour.

**Cas d'usage** :
- Conversations de recherche documentaire
- Analyse progressive de documents
- Q&A avec follow-up questions

***

## Partie 6 : Citations et Traçabilité

Un système RAG de qualité doit permettre de **vérifier les sources**.

In [19]:
def rag_with_citations(
    question: str,
    vector_store: VectorStore,
    k: int = 3
) -> Dict[str, Any]:
    """
    RAG avec génération structurée de citations.
    """
    # Retrieval
    query_embedding = create_embedding(question)
    chunks = vector_store.search(query_embedding, k=k)

    # Contexte numéroté pour citations
    context_parts = []
    for i, c in enumerate(chunks):
        context_parts.append(f"[{i+1}] {c['text'][:500]}")
    context = "\n\n".join(context_parts)

    # Prompt demandant des citations explicites
    prompt = f"""Basé sur ces sources:

{context}

Question: {question}

Instructions:
1. Réponds à la question en citant les sources entre crochets [1], [2], etc.
2. Si plusieurs sources confirment une information, cite-les tous.
3. Si l'information n'est pas dans les sources, indique-le.

Format de réponse:
RÉPONSE: [ta réponse avec citations]
SOURCES UTILISÉES: [liste des numéros de sources]
CONFIANCE: [haute/moyenne/basse]"""

    # Note: temperature not supported by gpt-5-mini, uses default (1.0)
    response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )

    return {
        "question": question,
        "response": response.choices[0].message.content,
        "retrieved_chunks": [
            {
                "id": i + 1,
                "score": c["score"],
                "text": c["text"],
                "preview": c["text"][:200],
            }
            for i, c in enumerate(chunks)
        ]
    }


# Test avec citations
result_cited = rag_with_citations(
    "What were the key points of disagreement between Lincoln and Douglas?",
    vector_store
)

print("=== RAG avec Citations ===")
print(f"\nQuestion: {result_cited['question']}")
print(f"\n{result_cited['response']}")
print("\n--- Chunks récupérés ---")
for chunk in result_cited['retrieved_chunks']:
    print(f"[{chunk['id']}] Score: {chunk['score']:.3f} - {chunk['preview'][:100]}...")

=== RAG avec Citations ===

Question: What were the key points of disagreement between Lincoln and Douglas?

RÉPONSE:
- Autorité nationale vs souveraineté des États — Douglas accuse Lincoln (et le parti républicain) de prôner « l’uniformité parmi les institutions des différents États » (idée nouvelle, selon Douglas) au lieu de respecter le principe de souveraineté populaire laissant « chaque État faire comme il l’entend » ; Douglas oppose donc sa doctrine de souveraineté populaire à ce qu’il appelle la doctrine d’uniformité de Lincoln [3].  
- Statut juridique de l’esclavage — le débat porte aussi sur l’idée d’étendre ou de rendre « pareil » le statut de l’esclavage dans tous les États (« advance, to make slavery alike lawful in all the States… »), ce point étant discuté dans les répliques de Douglas [2][3].  
- Accusations personnelles et caractère radical du parti républicain — Douglas reproche à Lincoln d’avoir voulu « abolitionizer » (rendre abolitionniste) les partis Whig et démoc

In [20]:
# Verification de la structure des citations
def verify_citations(result):
    """Verifie les citations."""
    response = result.get('response', '')
    chunks = result.get('retrieved_chunks', [])
    import re
    cited = set(re.findall(r'\[' + r'(\d+)\]', response))
    if not cited:
        return {"verdict": "affirmation non etayee", "details": "Aucune reference citee"}
    if not chunks:
        return {"verdict": "contexte vide", "details": "Aucun chunk"}
    valid = {str(c['id']) for c in chunks}
    invalid = cited - valid
    if invalid:
        return {"verdict": "reference inexistante", "details": "Refs " + str(invalid) + " non presentes"}
    return {"verdict": "structure valide", "details": "Toutes les refs OK"}

verification = verify_citations(result_cited)
print("\n=== Verification des citations ===")
print('Verdict:', verification['verdict'])
print('Details:', verification['details'])


=== Verification des citations ===
Verdict: structure valide
Details: Toutes les refs OK


### Analyse de la qualité des citations

Cette exécution démontre un système RAG où les citations sont **demandées**, puis soumises à une vérification structurelle.

**Résultats observés** :

| Aspect | Valeur | Interprétation |
|--------|--------|----------------|
| **Chunks récupérés** | 3 | `k=3` par défaut |
| **Références citées** | [1], [2], [3] | Les trois identifiants récupérés apparaissent dans la réponse |
| **Scores** | 0.673, 0.599, 0.595 | Similarité du retrieval, pas score de fidélité de la réponse |
| **Vérification automatique** | `structure valide` | Toutes les références citées correspondent à un chunk existant |

La fonction `verify_citations` valide uniquement la **structure** : présence d'au moins une référence, existence de chunks et absence d'identifiant hors ensemble. Elle ne prouve ni que chaque affirmation est étayée par le chunk associé, ni que la confiance « haute » produite par le modèle est calibrée.

**Vérification ponctuelle de contenu** :

- **Affirmation** : Douglas accuse Lincoln d'avoir voulu « abolitionize » les partis Whig et démocrate.
- **Source [1]** : l'extrait récupéré porte bien cette accusation.
- **Portée** : ce contrôle d'une affirmation ne généralise pas la fidélité à toute la réponse.

La prochaine étape pédagogique consiste donc à comparer manuellement plusieurs affirmations aux textes complets des chunks, puis à mesurer la précision et le rappel du retrieval sur un jeu de références.

### Exemple guidé 3 - Vérification du contenu des citations (EX-0422, crédité #21046)

**Objectif** : distinguer deux questions différentes pour une affirmation citée : est-elle appuyée par l'extrait effectivement transmis au modèle, puis est-elle appuyée ailleurs dans la source complète du chunk ?

**Méthode** :

1. Extraire une affirmation de la réponse avec sa référence, par exemple `[1]`.
2. Retrouver le dictionnaire dont l'`id` correspond dans `result_cited['retrieved_chunks']`.
3. Reconstituer le **contexte exact transmis au modèle** avec `text[:500]`. Le champ `text` conserve séparément la source complète du chunk et `preview` n'est qu'un aperçu d'affichage.
4. Comparer d'abord l'affirmation aux 500 caractères transmis et rendre un verdict explicite : `étayée`, `non étayée` ou `ambiguë` **dans le contexte fourni au modèle**.
5. Inspecter ensuite la source complète. Si elle contient un appui absent des 500 premiers caractères, le noter comme `étayée par la source complète, mais non par le contexte transmis` : ce passage ne faisait pas partie des éléments donnés au modèle pour cette génération.

**Exemple avec la réponse ci-dessus** :

```python
source_1 = next(
    chunk for chunk in result_cited["retrieved_chunks"]
    if chunk["id"] == 1
)
contexte_transmis_1 = source_1["text"][:500]
source_complete_1 = source_1["text"]

print("=== Contexte réellement transmis ===")
print(contexte_transmis_1)
print("\n=== Source complète du chunk ===")
print(source_complete_1)
```

Un score de similarité cosinus élevé indique une proximité entre la question et le chunk ; il ne certifie pas que le passage transmis étaye une affirmation particulière. Le modèle peut citer la mauvaise référence ou tirer une conclusion plus forte que son contexte. Inversement, trouver l'information seulement après le 500e caractère dans la source complète ne prouve pas que le modèle disposait de cette information.

> **Exemple guidé (crédité #21046).** La résolution ci-dessous (EX-0422, groupe de la PR #21046) contrôle l'affirmation [1] de la réponse archivée contre une source NPS reconstituée : localisation de la preuve, verdicts séparés et portée déclarée (aucun rejeu des embeddings ni de la génération). L'exercice EX-0422bis qui la suit applique le même protocole à une affirmation où les deux verdicts peuvent **diverger**.

In [21]:
# EX-0422 : contrôle reproductible avec assistance de ChatGPT.
# Réponse : sortie enregistrée dans le notebook du professeur, cellule rag_with_citations.
# Fragment : reconstitué depuis la page NPS récupérée, premier chunk de 400 mots.
# Ce contrôle ne prétend pas rejouer les embeddings ni la génération historique.
import re

reponse_archivee = "Les extraits montrent que les points de désaccord principaux entre Lincoln et Douglas portaient sur la question de l’esclavage et sur la légitimité de l’action politique républicaine.\n\n- Sur la politique à suivre pour l’esclavage : Douglas accuse Lincoln et les Républicains de vouloir imposer une « uniformité » des institutions (c.-à-d. une politique nationale sur l’esclavage) au lieu de respecter le droit de chaque État à décider (la souveraineté populaire) — accusation formulée explicitement par Douglas contre la « nouvelle doctrine » que, selon lui, Lincoln préconise [3]. Douglas reproche aussi à Lincoln (ou à sa position) de vouloir « rendre l’esclavage également licite dans tous les États — anciens comme nouveaux, Nord comme Sud » (formulation rapportée dans le débat) [2].\n\n- Sur le rôle et la nature du parti républicain / de Lincoln : Douglas accuse Lincoln d’avoir cherché à « abolitioniser » les partis Whig et démocrate et d’avoir participé, en 1854, à la rédaction d’une plate-forme républicaine très radicale — c’est une attaque contre les intentions politiques et l’antécédent organisationnel de Lincoln et du parti [1][2].\n\n- Attaques personnelles et d’ordre historique : Douglas reproche aussi à Lincoln d’avoir pris, selon lui, le « parti de l’ennemi » pendant la guerre du Mexique (une accusation portant sur la conduite passée de Lincoln) [1].\n\nRemarque : les extraits fournis rendent surtout compte des accusations et des thèmes du débat du côté de Douglas. Les arguments détaillés de Lincoln en réponse (sa défense des principes républicains, ses réfutations des accusations de Douglas, ou d’autres positions précises comme le traitement du Compromis de 1850 ou la décision Dred Scott) ne sont pas présents dans ces extraits, donc ne peuvent être confirmés ici à partir des sources fournies.\n\nSOURCES UTILISÉES: [1], [2], [3]\nCONFIANCE: haute"
source_reconstituee = "First Debate: Ottawa, Illinois August 21, 1858 It was dry and dusty, between 10,000 and 12,000 people were in attendance when the debate began at 2:00 p.m. There were no seats or bleachers. Douglas charged Lincoln with trying to “abolitionize” the Whig and Democratic Parties. He also charged Lincoln had been present when a very radical “abolitionist” type platform had been written by the Republican Party in 1854. Douglas accused Lincoln of taking the side of the common enemy in the Mexican War. Douglas also said Lincoln wanted to make Illinois “a free Negro colony.” Douglas asked Lincoln seven questions. Lincoln during his turn did not respond to the questions and was on the defensive denying the allegations Douglas had made. Lincoln charged Douglas with trying to nationalize slavery. In his rebuttal Douglas concentrated on the charge that Lincoln had been present when a very radical “abolitionist” type platform had been written by the Republican Party in 1854. Source: Neely, Mark E. Jr. 1982. The Abraham Lincoln Encyclopedia . New York: Da Capo Press, Inc. Full text of the Ottawa Debate follows. Mr. Douglas' Speech Ladies and gentlemen: I appear before you to-day for the purpose of discussing the leading political topics which now agitate the public mind. By an arrangement between Mr. Lincoln and myself, we are present here to-day for the purpose of having a joint discussion, as the representatives of the two great political parties of the State and Union, upon the principles in issue between those parties and this vast concourse of people, shows the deep feeling which pervades the public mind in regard to the questions dividing us. Prior to 1854 this country was divided into two great political parties, known as the Whig and Democratic parties. Both were national and patriotic, advocating principles that were universal in their application. An old line Whig could proclaim his principles in Louisiana and Massachusetts alike. Whig principles had no boundary sectional line, they were not limited by the Ohio river, nor by the Potomac, nor by the line of the free and slave States, but applied and were proclaimed wherever the Constitution ruled or the American flag waved over the American soil. (Hear him, and three cheers.) So it was, and so it is with the great Democratic party, which, from the days of Jefferson until this period, has proven itself to be the historic party of"
contexte_reconstitue = source_reconstituee[:500]
preuve = 'Douglas charged Lincoln with trying to “abolitionize” the Whig and Democratic Parties.'

# Sélection d'une affirmation réellement présente et associée à [1].
affirmation = next(
    ligne.strip() for ligne in reponse_archivee.splitlines()
    if 'abolitioniser' in ligne.lower() and '[1]' in ligne
)
print('PROVENANCE : réponse enregistrée, source NPS reconstituée ; aucun nouvel appel API.')
print('Source : https://home.nps.gov/liho/learn/historyculture/debate1.htm')
print('Passage de la réponse enregistrée :', affirmation)
print('Sous-affirmation contrôlée : Douglas accuse Lincoln de vouloir abolitioniser les partis Whig et démocrate [1].')
print('\nCONTEXTE RECONSTITUÉ TRANSMIS (500 caractères) :')
print(contexte_reconstitue)
print('\nPREUVE ET LOCALISATION :')
print(preuve)
print('Position dans la source :', source_reconstituee.find(preuve))
print('Longueur de la preuve :', len(preuve))
if preuve not in contexte_reconstitue:
    raise ValueError('La preuve attendue manque du contexte reconstitué : analyse à reprendre.')
print('\nVerdict contexte reconstitué : étayée.')
print('Verdict source complète du chunk reconstitué : étayée.')
print('Justification : le texte attribue explicitement cette accusation à Douglas.')
print('Portée : cela étaye le fait que Douglas porte une accusation, pas sa vérité.')
print('Divergence : aucune pour cette affirmation et ce fragment.')
print('Limite : le fragment historique exact et les embeddings ne sont pas archivés ici.')
print('Une nouvelle génération doit être contrôlée séparément sur ses propres extraits.')


PROVENANCE : réponse enregistrée, source NPS reconstituée ; aucun nouvel appel API.
Source : https://home.nps.gov/liho/learn/historyculture/debate1.htm
Passage de la réponse enregistrée : - Sur le rôle et la nature du parti républicain / de Lincoln : Douglas accuse Lincoln d’avoir cherché à « abolitioniser » les partis Whig et démocrate et d’avoir participé, en 1854, à la rédaction d’une plate-forme républicaine très radicale — c’est une attaque contre les intentions politiques et l’antécédent organisationnel de Lincoln et du parti [1][2].
Sous-affirmation contrôlée : Douglas accuse Lincoln de vouloir abolitioniser les partis Whig et démocrate [1].

CONTEXTE RECONSTITUÉ TRANSMIS (500 caractères) :
First Debate: Ottawa, Illinois August 21, 1858 It was dry and dusty, between 10,000 and 12,000 people were in attendance when the debate began at 2:00 p.m. There were no seats or bleachers. Douglas charged Lincoln with trying to “abolitionize” the Whig and Democratic Parties. He also charged 

### EXERCICE EX-0422bis - Chercher la divergence : une affirmation hors contexte transmis

L'exemple guidé 3 a contrôlé l'affirmation [1] (« abolitioniser » les partis) : étayée dans les deux sens, aucune divergence. Le cas intéressant est l'autre.

**Objectif** : appliquer le protocole à deux verdicts à l'affirmation marquée `[2]` de la même réponse archivée — Douglas reproche à Lincoln de vouloir « rendre l'esclavage également licite dans tous les États — anciens comme nouveaux, Nord comme Sud » — et voir si les deux verdicts **divergent**.

**Méthode** :

1. Extraire de `reponse_archivee` la ligne portant le marqueur `[2]` qui contient cette formulation.
2. Chercher un appui dans `contexte_reconstitue` (les 500 premiers caractères transmis au modèle) : contient-il une charge de Lincoln contre Douglas sur l'esclavage national ?
3. Chercher dans `source_reconstituee` **complète** une phrase attribuant à Lincoln une charge de « nationalize slavery », et localiser sa position exacte (`str.find`).
4. Rendre les deux verdicts séparés. Si l'appui se trouve après le 500e caractère : verdict « étayée par la source complète, mais non par le contexte transmis » — le modèle **ne disposait pas** de ce passage lors de la génération.

**Indices** :
- `# Étape 1` : filtrer les lignes de `reponse_archivee.splitlines()` sur `'licite'` et `'[2]'`
- `# Étape 2` : inspecter `contexte_reconstitue` ; une absence d'appui est un verdict à part entière (`non étayée dans le contexte transmis`)
- `# Étape 3` : `source_reconstituee.find("nationalize slavery")` donne la position de l'appui ; comparer à 500
- `# Indice` : notez aussi la **direction** de la charge — qui accuse qui — avant de conclure ; la formulation de la réponse archivée est une paraphrase, pas une citation

In [22]:
# EX-0422bis : chercher la divergence - l'affirmation [2] hors contexte transmis
# TODO etudiant : rendre les deux verdicts separes pour l'affirmation [2]
# Etape 1 : extraire de reponse_archivee la ligne portant 'licite' et '[2]'
# Etape 2 : chercher un appui dans contexte_reconstitue (500 premiers caracteres)
# Etape 3 : chercher "nationalize slavery" dans source_reconstituee complete,
#           relever position_appui_complet via str.find et comparer a 500
# Indice : si l'appui est au-dela du 500e caractere, les deux verdicts divergent
ligne_affirmation_2 = None  # TODO etudiant
verdict_contexte_transmis = None  # TODO etudiant
verdict_source_complete = None  # TODO etudiant
position_appui_complet = None  # TODO etudiant
print("EX-0422bis a completer : les deux verdicts de l'affirmation [2]")

EX-0422bis a completer : les deux verdicts de l'affirmation [2]


In [23]:
# Exemple de filtrage par score (aucun réordonnancement, conservation de l'ordre existant)
def rag_with_score_filtering(
    question: str,
    vector_store: VectorStore,
    k: int = 5,
    min_score: float = 0.3
) -> Dict[str, Any]:
    """
    RAG avec filtrage par score: filtre les chunks peu pertinents.
    """
    query_embedding = create_embedding(question)
    all_chunks = vector_store.search(query_embedding, k=k)

    # Filtrage par score minimum
    filtered_chunks = [c for c in all_chunks if c['score'] >= min_score]

    print(f"Chunks récupérés: {len(all_chunks)}, après filtrage (score >= {min_score}): {len(filtered_chunks)}")

    if not filtered_chunks:
        return {"answer": "Aucun contexte pertinent trouvé pour cette question."}

    # Génération avec chunks filtrés
    context = "\n\n".join([c['text'] for c in filtered_chunks])

    response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=[
            {"role": "system", "content": "Réponds uniquement basé sur le contexte fourni."},
            {"role": "user", "content": f"Contexte:\n{context}\n\nQuestion: {question}"}
        ]
    )

    return {
        "answer": response.choices[0].message.content,
        "chunks_used": len(filtered_chunks),
        "chunks_filtered_out": len(all_chunks) - len(filtered_chunks)
    }


# Test du filtrage par seuil
result_filtered = rag_with_score_filtering(
    "What was the outcome of the debate?",
    vector_store,
    k=5,
    min_score=0.3
)

print(f"\nRéponse: {result_filtered['answer'][:500]}...")

Chunks récupérés: 5, après filtrage (score >= 0.3): 5



Réponse: Le texte ne donne pas de « vainqueur » officiel. La rencontre se termina après les discours et les répliques : Lincoln prononça son long discours (accueilli par de vifs applaudissements — environ les deux tiers de l'audience), puis Douglas fit sa réponse en insistant sur l'accusation concernant la plateforme de 1854. Aucun verdict ou résultat électoral n'est indiqué dans le passage fourni....


### Exercice 3 - Evaluation de la qualite de retrieval

Le pipeline RAG depend entierement de la qualite du retrieval. Sans bons chunks, même le meilleur modèle génèrera des reponses médiocre.

**Objectif** : Implementer une fonction `evaluate_retrieval(questions_références, vector_store, k)` qui mesure la precision du retrieval sur un jeu de questions avec chunks de référence attendus.

**Indices** :
- Le paramètre `questions_références` est une liste de dicts `{"question": str, "expected_chunk_ids": List[int]}`
- Pour chaque question, effectuer la recherche k-NN et vérifier si les `expected_chunk_ids` apparaissent dans les résultats
- Calculer deux metriques : **precision** (fraction des chunks trouves qui sont pertinents) et **recall** (fraction des chunks pertinents qui ont ete trouves)
- `# Étape 1` : Construire un petit jeu de 3 questions avec chunks attendus (utiliser les chunks 0, 9, 20 par exemple)
- `# Étape 2` : Pour chaque question, appeler `vector_store.search()` et collecter les `chunk_id` trouves
- `# Étape 3` : Comparer avec les `expected_chunk_ids` et calculer precision et recall
- `# Indice` : precision = `len(trouves & attendus) / len(trouves)`, recall = `len(trouves & attendus) / len(attendus)`

In [24]:
# Exercice 3 - Evaluation de la qualite de retrieval
# TODO etudiant : implementer evaluate_retrieval

def evaluate_retrieval(questions_references: List[Dict], vector_store: VectorStore, k: int = 3) -> Dict[str, float]:
    """
    Evalue la qualite du retrieval sur un jeu de questions de reference.
    
    Args:
        questions_references: Liste de dicts {"question": str, "expected_chunk_ids": List[int]}
        vector_store: Base vectorielle initiee
        k: Nombre de chunks a recuperer par recherche
    
    Returns:
        Dict avec precision_moyenne, recall_moyen, details_par_question
    """
    pass  # TODO etudiant : implementer l'evaluation du retrieval

# Etape 1 : Definir un jeu de 3 questions avec chunks attendus
# questions_references = [
#     {"question": "What did Lincoln say about a house divided?", "expected_chunk_ids": [9]},
#     {"question": "What was the audience size?", "expected_chunk_ids": [0]},
#     {"question": "What did Douglas say about popular sovereignty?", "expected_chunk_ids": [20, 31]},
# ]

# Etape 2 : Appeler evaluate_retrieval
# results = evaluate_retrieval(questions_references, vector_store, k=3)

# Etape 3 : Afficher les resultats
# print(f"Precision moyenne: {results['precision_moyenne']:.2f}")
# print(f"Recall moyen: {results['recall_moyen']:.2f}")

print("Exercice a completer")

Exercice a completer


### Analyse du filtrage et gestion des limites

Cette exécution illustre deux aspects importants : le filtrage par score et la gestion des questions dont la réponse n'est pas explicitement formulée dans le contexte.

**Résultats du filtrage** :

| Métrique | Valeur | Interprétation |
|----------|--------|----------------|
| **Chunks récupérés (k=5)** | 5 | Recherche k-NN standard |
| **Chunks après filtrage (score ≥ 0.3)** | 5 | Tous les chunks conservés |
| **Chunks filtrés** | 0 | Aucun chunk avec score < 0.3 |

**Observation** : avec ce seuil et cette requête, les cinq résultats sont conservés. Cette mesure ne suffit pas à conclure que chaque chunk est pertinent ; elle montre seulement qu'aucun score n'est inférieur à 0.3.

**Synthèse non verbatim de la réponse** : le texte ne désigne pas de vainqueur formel. Il décrit la fin du débat et une réaction publique favorable à Lincoln, sans transformer cet indice en résultat officiel.

**Points forts observés** :

1. **Prudence** : le modèle distingue l'absence de verdict formel de la réaction du public.
2. **Contexte fourni** : il répond à partir des éléments disponibles plutôt que d'inventer un vainqueur.
3. **Limite explicite** : la réponse ne présente pas l'indice d'audience comme un résultat définitif.

Les seuils de similarité ne sont pas universels : ils doivent être calibrés sur un jeu de questions annoté et sur le compromis précision/rappel du cas d'usage. L'exercice précédent propose justement de mesurer ce compromis au lieu d'adopter une table de seuils génériques.

**Prochaine étape** : évaluer plusieurs seuils sur le jeu de références, puis comparer précision, rappel et qualité des réponses générées.

***

## Conclusion

### Points clés

1. **Chunking** : Stratégie critique - fixe, sémantique, ou récursif selon le cas
2. **Embeddings** : `text-embedding-3-large` offre la meilleure précision
3. **Retrieval** : k-NN avec score minimum pour filtrer le bruit
4. **Responses API** : `previous_response_id` pour multi-turn efficace
5. **Citations** : Toujours demander des références vérifiables

### Prochaines étapes

- **Notebook 6** : PDF et Web Search intégrés
- **Notebook 7** : Code Interpreter pour analyse de données
- **Notebook 9** : Patterns de production (batch, retry, monitoring)

### Ressources

**Références académiques :**
- **Lewis, P. et al. (2020)** - *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. arXiv:2005.11401. (RAG originel, nomme et définit le paradigme.)
- **Karpukhin, V. et al. (2020)** - *Dense Passage Retrieval for Open-Domain Question Answering*. arXiv:2004.04906. (Retriever dense bi-encoder.)
- **Reimers, N. & Gurevych, I. (2019)** - *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks*. arXiv:1908.10084. (Embeddings de phrases.)
- **Es, S., James, J., Espinosa-Anke, L., & Schockaert, S. (2023)** - *RAGAS: Automated Evaluation of Retrieval Augmented Generation*. arXiv:2309.15217. (Évaluation RAG : faithfulness, context precision.)

**Documentation API :**
- [OpenAI Embeddings Guide](https://platform.openai.com/docs/guides/embeddings)
- [Responses API Documentation](https://platform.openai.com/docs/api-référence/responses)
- [RAG Best Practices](https://platform.openai.com/docs/guides/retrieval)

In [25]:
# Résumé des fonctions créées
print("=== Fonctions RAG disponibles ===")
print("""
Chunking:
  - chunk_fixed(text, chunk_size, overlap)
  - chunk_semantic(text, max_sentences)
  - chunk_recursive(text, max_size, delimiters)

Embeddings:
  - create_embedding(text, model)
  - create_embeddings_batch(texts, model)

Recherche:
  - VectorStore.search(query_embedding, k)

RAG:
  - rag_query(question, vector_store, k)           # Chat Completions
  - rag_query_responses_api(question, ...)         # Responses API (multi-turn)
  - rag_with_citations(question, vector_store, k)  # Avec sources
  - rag_with_score_filtering(question, ..., min_score)   # Avec filtrage
""")

=== Fonctions RAG disponibles ===

Chunking:
  - chunk_fixed(text, chunk_size, overlap)
  - chunk_semantic(text, max_sentences)
  - chunk_recursive(text, max_size, delimiters)

Embeddings:
  - create_embedding(text, model)
  - create_embeddings_batch(texts, model)

Recherche:
  - VectorStore.search(query_embedding, k)

RAG:
  - rag_query(question, vector_store, k)           # Chat Completions
  - rag_query_responses_api(question, ...)         # Responses API (multi-turn)
  - rag_with_citations(question, vector_store, k)  # Avec sources
  - rag_with_score_filtering(question, ..., min_score)   # Avec filtrage



***

## CHALLENGE BONUS - Mini Moteur FAQ avec RAG

**Points : 0.5 pts**

### Objectif

Créer un mini système RAG qui répond aux questions en cherchant dans une base de connaissances.

### Ce que vous avez appris

Ce notebook montre:
- **Section 2** : Chunking avec `chunk_recursive()`
- **Section 3** : Embeddings avec `client.embeddings.create()`
- **Section 4** : Recherche vectorielle avec la classe `VectorStore`
- **Section 5** : Construction d'un prompt RAG avec contexte

### Critères de succès

- [ ] Créer une mini-base de 5-8 Q/R sur un sujet de votre choix
- [ ] Implémenter le chunking des réponses avec `chunk_recursive()`
- [ ] Créer les embeddings avec `text-embedding-3-small`
- [ ] Implémenter la recherche par similarité (réutiliser `cosine_similarity`)
- [ ] Construire un prompt RAG qui injecte le contexte
- [ ] Tester avec une question et afficher la réponse + sources

### Contraintes techniques

- Réutiliser les fonctions du notebook (`chunk_recursive`, `VectorStore`)
- Afficher les sources (chunks) utilisées pour la réponse

***

**Soumission** : PR avec titre "Challenge #5 - [Votre Nom]", sujet de la FAQ et exemple de requête
